## VDJtoMZratio

In [6]:
import pandas as pd

def VDJtoMZratio(input_file, output_name):
    ## Load files (c_df: constant region, s_df: signal peptide, a_df: amino acid weight)
    c_df = pd.read_csv('99.references/constant.regions.tsv', sep="\t")
    codon_df = pd.read_csv('99.references/codon.txt', sep="\t")
    s_df = pd.read_csv('99.references/signal.peptides.Vgenes.csv', sep=",")
    a_df = pd.read_csv('99.references/AA.weight2.csv', sep=",")
    df = pd.read_csv(input_file, sep=",")
    
    ## Create dictionaries for constant region and signal peptide for filtering
    c_dict = dict(zip(c_df['C.gene'], c_df['AA.constant']))
    s_dict = dict(zip(s_df['Gene'], s_df['Signal peptide']))
    a_dict = dict(zip(a_df['Amino.acid.name.1.letter'], a_df['Average.Weight.Dalton']))
    light_jc_junc_dict = dict(zip(c_df['C.gene'], c_df['nt.constant'].astype(str).str[:2]))
    codon_dict = dict(zip(codon_df['Codon'], codon_df['Letter']))

    ## Filter contig to have high confident / intact cell 
    cond1 = (df['is_cell'] == True)
    cond2 = (df['high_confidence'] == True)
    cond3 = (df['full_length'] == True)
    cond4 = (df['productive'] == True)
    df = df[ cond1 & cond2 & cond3 & cond4]

    ## Apply dictionaries
    df['signal_peptide'] = df['v_gene'].map(s_dict)
    df['c_aa_pre'] = df['c_gene'].map(c_dict)

    ## Get VDJ nt and aa sequence
    df['vdj_nt'] = df['fwr1_nt'] + df['cdr1_nt'] + df['fwr2_nt'] + df['cdr2_nt'] + df['fwr3_nt'] + df['cdr3_nt'] + df['fwr4_nt']
    df['vdj_aa'] = df['fwr1'] + df['cdr1'] + df['fwr2'] + df['cdr2'] + df['fwr3'] + df['cdr3'] + df['fwr4']

    ## Check J-C junction SNP/mutation, and assign appropriate AA
    df['light_jc_junc_2nt'] = df['c_gene'].map(light_jc_junc_dict)
    df['light_jc_junc_full_nt'] = df['vdj_nt'].astype(str).str[-1] + df['light_jc_junc_2nt'].astype(str)
    df['light_jc_aa'] = df['light_jc_junc_full_nt'].map(codon_dict)
    df['c_aa'] = df['light_jc_aa'] + df['c_aa_pre']
    
    ## Reconstruct AA sequence final
    df['full_aa_seq'] = df['vdj_aa'] + df['c_aa']
    
    ## heavy CDR3 (including heavy chain)
    heavy_df = df[df['chain'] == 'IGH']
    heavy_df_sorted = heavy_df.sort_values('umis', ascending=False) ## get contig with most UMIs
    heavy_map = heavy_df_sorted.drop_duplicates(subset='barcode', keep='first')
    heavy_map_aa = heavy_map.set_index('barcode')['cdr3']
    heavy_map_nt = heavy_map.set_index('barcode')['cdr3_nt']
    df['heavy_cdr3_aa'] = df['barcode'].map(heavy_map_aa)
    df['heavy_cdr3_nt'] = df['barcode'].map(heavy_map_nt)
    
    ## Remove signal peptide
    df['full_aa_seq_filt'] = df.apply(lambda x: 
                x['full_aa_seq'].replace(str(x['signal_peptide']), '') 
                if pd.notna(x['full_aa_seq']) and pd.notna(x['signal_peptide']) 
                else x['full_aa_seq'], axis=1)

    ## Heavy chain dictionary
    h_df = df[df['chain'] == 'IGH']
    h_dict = dict(zip(h_df['barcode'], h_df['c_gene']))

    ## Use only light chain (skip contig_1, IGH)
    df = df[df['chain'] != 'IGH']

    ## Calculate weight and apply
    def calculate_weight(sequence, mapping_dict):
        return sum(mapping_dict.get(char, 0) for char in sequence)

    df = df[df['full_aa_seq_filt'].notnull()]

    df['weight'] = df['full_aa_seq_filt'].apply(lambda seq:
                                               calculate_weight(seq, a_dict))

    ## For two light chain contig, take the contig with more UMI
    idx = df.groupby('barcode')['umis'].idxmax()
    df = df.loc[idx].reset_index(drop=True)
    
    ## light CDR3
    df['light_cdr3_aa'] = df['cdr3']
    df['light_cdr3_nt'] = df['cdr3_nt']

    ## Calculate m/z ratio (add water molecule and divide by electron 2)
    df['mz_ratio'] = (df['weight'].astype(float) + float(18))/2

    ## get heavy chain constant gene
    df['heavy_c_gene_all_contig'] = df['barcode'].map(h_dict)

    ## get clonality (count the number of same clonotype)
    df['clonal'] = df['raw_clonotype_id'].transform(lambda x: df['raw_clonotype_id'].value_counts().get(x))
    
    df = df[['barcode','heavy_c_gene_all_contig', 
             'v_gene', 'j_gene', 'c_gene', 'full_aa_seq_filt', 
             'heavy_cdr3_aa', 'light_cdr3_aa',
             'heavy_cdr3_nt', 'light_cdr3_nt',
             'vdj_nt',
             'weight', 'mz_ratio', 'clonal']]

    ## Write output
    df.to_csv(f'{output_name}', sep=",", index=False)


## Apply VDJtoMZratio (Example)

In [9]:
import pandas as pd
import subprocess 

VDJtoMZratio('example/all_contig_annotations.csv', 'example/output.csv')

